# GMM Behavioral Regime Detection — C2C Marketplace
**Capstone Project | Author: Arevik Melikyan | Supervisor: Varazdat Stepanyan**

## 0. Imports

In [13]:
import warnings
import os
import json
from pathlib import Path
from glob import glob
from datetime import datetime

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import RobustScaler, StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score,
    adjusted_rand_score,
)
from sklearn.impute import SimpleImputer

import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from tqdm.notebook import tqdm

try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("[INFO] pip install umap-learn for UMAP visualisation")

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

sns.set_theme(style="whitegrid", font_scale=1.1)
PALETTE = ["#4361EE", "#F72585", "#4CC9F0", "#7209B7",
           "#3A0CA3", "#560BAD", "#480CA8", "#B5179E"]
plt.rcParams.update({"figure.dpi": 120,
                     "axes.spines.top": False,
                     "axes.spines.right": False})

DATASET_ROOT = Path(os.environ.get("DATASET_ROOT", "/Volumes/T5 EVO"))
DATASET_PATH = DATASET_ROOT / "hf" / "merrec"
assert DATASET_PATH.exists(), f"Not found: {DATASET_PATH}"
print(f"Ready | {datetime.now():%Y-%m-%d %H:%M:%S}")
print(f"Dataset: {DATASET_PATH}")

Ready | 2026-03-21 13:00:58
Dataset: /Volumes/T5 EVO/hf/merrec


## 1. Configuration

In [14]:
CFG = {
    # ── Columns ───────────────────────────────────────────────────────────────
    "col_user":            "user_id",
    "col_item":            "item_id",
    "col_action":          "event_id",
    "col_time":            "stime",
    "positive_action":     "item_view",

    # ── Streaming limit (absolute integer, same key as NFM) ───────────────────
    "max_total_rows":      2_000,

    # ── Session boundary ──────────────────────────────────────────────────────
    "session_gap_s":       1800,          # 30 min gap = new session
    "min_session_events":  2,             # drop single-event sessions

    # ── Features for GMM ──────────────────────────────────────────────────────
    "features": [
        "session_length_s",
        "n_events",
        "n_unique_items",
        "revisit_rate",
        "inter_event_time_median",
        "session_velocity",
    ],

    # ── Preprocessing ─────────────────────────────────────────────────────────
    "scaler":              "robust",
    "log_transform_cols": [
        "session_length_s",
        "n_events",
        "n_unique_items",
        "inter_event_time_median",
    ],

    # ── GMM grid search ───────────────────────────────────────────────────────
    "k_range":             range(2, 9),
    "covariance_types":    ["full", "tied", "diag", "spherical"],
    "n_init":              10,
    "max_iter":            500,
    "tol":                 1e-4,

    # ── Final model ───────────────────────────────────────────────────────────
    "final_k":             None,          # None = auto via BIC
    "final_cov_type":      "full",

    # ── Stability ─────────────────────────────────────────────────────────────
    "bootstrap_n":         30,
    "bootstrap_sample_frac": 0.8,
    "silhouette_sample_size": 5_00,

    # ── Output ────────────────────────────────────────────────────────────────
    "output_dir":          "gmm_outputs",
}

Path(CFG["output_dir"]).mkdir(parents=True, exist_ok=True)
print("CFG loaded.")
for k, v in CFG.items():
    print(f"  {k:35s}: {v}")

CFG loaded.
  col_user                           : user_id
  col_item                           : item_id
  col_action                         : event_id
  col_time                           : stime
  positive_action                    : item_view
  max_total_rows                     : 2000
  session_gap_s                      : 1800
  min_session_events                 : 2
  features                           : ['session_length_s', 'n_events', 'n_unique_items', 'revisit_rate', 'inter_event_time_median', 'session_velocity']
  scaler                             : robust
  log_transform_cols                 : ['session_length_s', 'n_events', 'n_unique_items', 'inter_event_time_median']
  k_range                            : range(2, 9)
  covariance_types                   : ['full', 'tied', 'diag', 'spherical']
  n_init                             : 10
  max_iter                           : 500
  tol                                : 0.0001
  final_k                            : None
  fi

## 2. Streaming Class

```
MerRecSessionStreamer
│
├── __init__   — discover parquet files, store cfg
├── _iter_files — sorted file list (same as NFM)
└── __iter__   — PyArrow batch loop, row-by-row processing
                 identical structure to MerRecStreamingCTR.__iter__
                 yields one session-feature dict per completed session
```

**Memory model — O(1) per user:**  
`user_state[u]` holds only the running accumulators for the user's **current** session.  
When a session boundary (`gap > session_gap_s`) is hit inline, the completed session is  
yielded immediately and state is reset. Nothing is buffered at file level.

In [15]:
class MerRecSessionStreamer:
    """
    Streams Parquet files and yields one session-feature dict per
    completed session.  Mirrors MerRecStreamingCTR.__iter__ exactly.

    Fix: `done` flag breaks out of the FILE loop immediately after
    max_total_rows is hit — the next file is never opened.
    """

    def __init__(self, dataset_path: Path, cfg: dict):
        self.dataset_path = Path(dataset_path)
        self.cfg          = cfg

        self.files = sorted(
            glob(str(self.dataset_path / "**" / "*.parquet"), recursive=True)
        )
        if not self.files:
            raise FileNotFoundError(
                f"No parquet files found under {self.dataset_path}"
            )
        print(f"[MerRecSessionStreamer] {len(self.files)} parquet files found")

    # ------------------------------------------------------------------
    def _iter_files(self) -> list:
        return self.files[:]

    # ------------------------------------------------------------------
    @staticmethod
    def _new_state(ts: float, item: str) -> dict:
        return {
            "session_start": ts,
            "last_ts":       ts,
            "n_events":      1,
            "item_counts":   {item: 1},
            "iets":          [],
        }

    # ------------------------------------------------------------------
    @staticmethod
    def _emit(user_id: str, state: dict) -> dict:
        n_events    = state["n_events"]
        item_counts = state["item_counts"]
        n_unique    = len(item_counts)
        duration    = state["last_ts"] - state["session_start"]

        revisit_rate = (
            sum(1 for c in item_counts.values() if c > 1) / n_unique
            if n_unique > 0 else 0.0
        )
        iets       = state["iets"]
        iet_median = float(np.median(iets)) if iets else 0.0
        velocity   = n_events / max(duration, 1.0)

        return {
            "user_id":                 user_id,
            "session_length_s":        float(duration),
            "n_events":                n_events,
            "n_unique_items":          n_unique,
            "revisit_rate":            revisit_rate,
            "inter_event_time_median": iet_median,
            "session_velocity":        float(velocity),
        }

    # ------------------------------------------------------------------
    def __iter__(self):
        col_u      = self.cfg["col_user"]
        col_i      = self.cfg["col_item"]
        col_a      = self.cfg["col_action"]
        col_t      = self.cfg["col_time"]
        pos_action = self.cfg["positive_action"]
        gap_s      = self.cfg["session_gap_s"]
        min_events = self.cfg["min_session_events"]
        max_total  = self.cfg["max_total_rows"]

        user_state = {}
        total_rows = 0
        done       = False          # ← stops next file being opened

        for fp in self._iter_files():

            if done:                # ← checked BEFORE ds.dataset() is called
                break

            try:
                dataset      = ds.dataset(fp, format="parquet")
                schema_names = dataset.schema.names

                item_col = col_i if col_i in schema_names else "product_id"
                if item_col not in schema_names:
                    print(f"[WARN] no item column in {fp}, skipping")
                    continue
                if col_t not in schema_names:
                    print(f"[WARN] no timestamp column in {fp}, skipping")
                    continue

                scanner = dataset.scanner(
                    columns=[col_u, item_col, col_a, col_t],
                    batch_size=65536,
                )

                for record_batch in scanner.to_batches():

                    users   = record_batch.column(col_u).to_pylist()
                    items   = record_batch.column(item_col).to_pylist()
                    actions = record_batch.column(col_a).to_pylist()
                    times   = record_batch.column(col_t).to_pylist()

                    for u, it, act, ts_raw in zip(users, items, actions, times):

                        if total_rows >= max_total:
                            for uid, st in user_state.items():
                                if st["n_events"] >= min_events:
                                    yield self._emit(uid, st)
                            done = True     # ← set flag
                            break           # ← break row loop

                        if act != pos_action:
                            continue
                        if u is None or it is None or ts_raw is None:
                            continue

                        try:
                            ts = float(ts_raw)
                        except (TypeError, ValueError):
                            continue

                        total_rows += 1
                        it_str = str(it)

                        if u not in user_state:
                            user_state[u] = self._new_state(ts, it_str)
                        else:
                            st  = user_state[u]
                            gap = ts - st["last_ts"]

                            if gap > gap_s:
                                if st["n_events"] >= min_events:
                                    yield self._emit(u, st)
                                user_state[u] = self._new_state(ts, it_str)
                            else:
                                st["iets"].append(gap)
                                st["last_ts"]  = ts
                                st["n_events"] += 1
                                st["item_counts"][it_str] = (
                                    st["item_counts"].get(it_str, 0) + 1
                                )

                    if done:        # ← break batch loop
                        break

            except Exception as e:
                print(f"[WARN] Failed reading {fp}: {e}")
                continue

        # Final flush — only reached if we exhausted all files naturally
        if not done:
            for uid, st in user_state.items():
                if st["n_events"] >= min_events:
                    yield self._emit(uid, st)


print("MerRecSessionStreamer defined.")

MerRecSessionStreamer defined.


## 3. Load Data via Streamer
> Iterate the class, collect session dicts, build DataFrame.

In [18]:
# ═══════════════════════════════════════════════════════════════
# CELL 3 — STREAM TO PARQUET (SCALABLE)
# ═══════════════════════════════════════════════════════════════

import pyarrow as pa
import pyarrow.parquet as pq

OUTPUT_PATH = Path(CFG["output_dir"]) / "sessions.parquet"

def stream_sessions_to_parquet(streamer, out_path, chunk_size=10_000):
    buffer = []
    writer = None
    total = 0

    for session in tqdm(streamer, desc="Streaming → Parquet"):
        buffer.append(session)

        if len(buffer) >= chunk_size:
            df = pd.DataFrame(buffer)
            table = pa.Table.from_pandas(df)

            if writer is None:
                writer = pq.ParquetWriter(out_path, table.schema)

            writer.write_table(table)
            total += len(buffer)
            buffer = []

    if buffer:
        df = pd.DataFrame(buffer)
        table = pa.Table.from_pandas(df)
        if writer is None:
            writer = pq.ParquetWriter(out_path, table.schema)
        writer.write_table(table)
        total += len(buffer)

    if writer:
        writer.close()

    print(f"\nSaved sessions → {out_path}")
    print(f"Total sessions: {total:,}")


# ── RUN STREAMING ──
streamer = MerRecSessionStreamer(DATASET_PATH, CFG)
stream_sessions_to_parquet(streamer, OUTPUT_PATH)

[MerRecSessionStreamer] 2170 parquet files found


Streaming → Parquet: 0it [00:00, ?it/s]

KeyboardInterrupt: 

In [19]:
# ═══════════════════════════════════════════════════════════════
# LOAD SAMPLE FOR MODELING
# ═══════════════════════════════════════════════════════════════

SAMPLE_SIZE = 100_000  # adjust (50k–500k safe)

dataset = ds.dataset(OUTPUT_PATH, format="parquet")

table = dataset.to_table(columns=CFG["features"] + ["user_id"])

df_sessions = table.to_pandas()

if len(df_sessions) > SAMPLE_SIZE:
    df_sessions = df_sessions.sample(SAMPLE_SIZE, random_state=RANDOM_STATE)

print(f"Sample loaded: {df_sessions.shape}")

FileNotFoundError: gmm_outputs/sessions.parquet

## 4. EDA

In [ ]:
print(f"Shape             : {df_sessions.shape}")
print(f"Missing values:")
print(df_sessions[CFG["features"]].isnull().sum().to_string())
print("\nDescriptive statistics:")
display(df_sessions[CFG["features"]].describe().round(3))

In [ ]:
n_feat = len(CFG["features"])
n_cols = 3
n_rows = -(-n_feat // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 3.5))
axes = axes.flatten()

for i, feat in enumerate(CFG["features"]):
    ax   = axes[i]
    data = df_sessions[feat].dropna()
    cap  = data.quantile(0.99)
    ax.hist(data[data <= cap], bins=60, color=PALETTE[0], alpha=0.75, edgecolor="none")
    ax.set_title(feat, fontsize=11, fontweight="bold")
    ax.set_ylabel("Count")
    skew = data.skew()
    ax.text(0.97, 0.95, f"skew={skew:.2f}\nn={len(data):,}",
            transform=ax.transAxes, ha="right", va="top", fontsize=8,
            color="firebrick" if abs(skew) > 1 else "gray")

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle("Feature distributions (raw, capped p99 for display)",
             fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/01_feature_distributions.png", bbox_inches="tight")
plt.show()
print("[SAVED] 01_feature_distributions.png")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Correlation heatmap
corr = df_sessions[CFG["features"]].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f",
            cmap="coolwarm", center=0, linewidths=0.5, ax=axes[0],
            cbar_kws={"shrink": 0.8})
axes[0].set_title("Feature correlation matrix", fontweight="bold")

# Skewness
skewness = df_sessions[CFG["features"]].skew().sort_values(ascending=False)
colors   = ["firebrick" if abs(s) > 1 else PALETTE[0] for s in skewness]
axes[1].barh(skewness.index, skewness.values, color=colors, alpha=0.8)
axes[1].axvline(1,  color="firebrick", lw=1.2, ls="--", alpha=0.6)
axes[1].axvline(-1, color="firebrick", lw=1.2, ls="--", alpha=0.6)
axes[1].axvline(0,  color="gray", lw=0.8)
axes[1].set_xlabel("Skewness")
axes[1].set_title("Feature skewness", fontweight="bold")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/02_eda.png", bbox_inches="tight")
plt.show()
print("[SAVED] 02_eda.png")

## 5. Preprocessing

In [16]:
def build_feature_matrix(df: pd.DataFrame, cfg: dict):
    features = cfg["features"]
    X = df[features].copy().astype(float)

    # Clip p99.5
    for col in features:
        X[col] = X[col].clip(upper=X[col].quantile(0.995))

    # Impute
    X = pd.DataFrame(
        SimpleImputer(strategy="median").fit_transform(X),
        columns=features
    )

    # log1p
    for col in cfg["log_transform_cols"]:
        if col in features:
            X[col] = np.log1p(X[col])

    # Scale
    Scaler = RobustScaler if cfg["scaler"] == "robust" else StandardScaler
    scaler = Scaler()
    return scaler.fit_transform(X), scaler


X, fitted_scaler = build_feature_matrix(df_sessions, CFG)
print(f"Feature matrix: {X.shape}")
print(f"Mean : {X.mean(axis=0).round(3)}")
print(f"Std  : {X.std(axis=0).round(3)}")

fig, axes = plt.subplots(2, 3, figsize=(16, 7))
for i, (ax, feat) in enumerate(zip(axes.flatten(), CFG["features"])):
    ax.hist(X[:, i], bins=60, color=PALETTE[1], alpha=0.75, edgecolor="none")
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.axvline(0, color="black", lw=0.8, ls="--")
fig.suptitle("Features after log1p + RobustScaler", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/03_features_scaled.png", bbox_inches="tight")
plt.show()
print("[SAVED] 03_features_scaled.png")

NameError: name 'df_sessions' is not defined

## 6. Optimal K Selection (BIC / AIC / Silhouette)

In [ ]:
def gmm_model_selection(X, cfg, random_state):
    results = []
    total   = len(list(cfg["k_range"])) * len(cfg["covariance_types"])
    pbar    = tqdm(total=total, desc="Grid search")

    for cov_type in cfg["covariance_types"]:
        for k in cfg["k_range"]:
            try:
                gmm = GaussianMixture(
                    n_components=k, covariance_type=cov_type,
                    n_init=cfg["n_init"], max_iter=cfg["max_iter"],
                    tol=cfg["tol"], random_state=random_state,
                )
                gmm.fit(X)
                labels   = gmm.predict(X)
                n_unique = len(np.unique(labels))

                if n_unique >= 2:
                    idx = np.random.choice(
                        len(X), min(cfg["silhouette_sample_size"], len(X)), replace=False
                    )
                    sil = silhouette_score(X[idx], labels[idx])
                    db  = davies_bouldin_score(X, labels)
                    ch  = calinski_harabasz_score(X, labels)
                else:
                    sil = db = ch = np.nan

                results.append({
                    "k": k, "cov_type": cov_type,
                    "bic": gmm.bic(X), "aic": gmm.aic(X),
                    "log_likelihood": gmm.lower_bound_,
                    "silhouette": sil, "davies_bouldin": db,
                    "calinski_harabasz": ch, "converged": gmm.converged_,
                })
            except Exception as e:
                print(f"  [WARN] k={k} cov={cov_type}: {e}")
            pbar.update(1)

    pbar.close()
    return pd.DataFrame(results).sort_values("bic").reset_index(drop=True)


selection_df = gmm_model_selection(X, CFG, RANDOM_STATE)
print("\nTop 10 by BIC:")
display(selection_df.head(10))
selection_df.to_csv(f"{CFG['output_dir']}/model_selection_results.csv", index=False)
print("[SAVED] model_selection_results.csv")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (metric, title, low) in zip(axes, [
    ("bic",        "BIC (lower = better)",          True),
    ("aic",        "AIC (lower = better)",          True),
    ("silhouette", "Silhouette (higher = better)",  False),
]):
    for j, cov in enumerate(CFG["covariance_types"]):
        sub = selection_df[selection_df["cov_type"] == cov].sort_values("k")
        ax.plot(sub["k"], sub[metric], marker="o", lw=2,
                color=PALETTE[j], label=cov, alpha=0.85)
    sub_full = selection_df[selection_df["cov_type"] == CFG["final_cov_type"]]
    if not sub_full.empty:
        best = sub_full.sort_values(metric, ascending=low).iloc[0]
        ax.axvline(best["k"], color="red", ls="--", lw=1.5,
                   label=f"Best k={int(best['k'])} (full)")
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("K")
    ax.set_xticks(list(CFG["k_range"]))
    ax.legend(fontsize=9)

fig.suptitle("GMM model selection", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/04_model_selection_curves.png", bbox_inches="tight")
plt.show()
print("[SAVED] 04_model_selection_curves.png")

In [ ]:
if CFG["final_k"] is None:
    sub  = selection_df[selection_df["cov_type"] == CFG["final_cov_type"]]
    best = sub.sort_values("bic").iloc[0]
    BEST_K = int(best["k"])
    print(f"[AUTO] Best K={BEST_K}  BIC={best['bic']:.1f}  silhouette={best['silhouette']:.4f}")
else:
    BEST_K = int(CFG["final_k"])
    print(f"[MANUAL] K={BEST_K}")

## 7. Final GMM Fitting

In [ ]:
print(f"Fitting final GMM  K={BEST_K}  cov='{CFG['final_cov_type']}'")
final_gmm = GaussianMixture(
    n_components=BEST_K, covariance_type=CFG["final_cov_type"],
    n_init=CFG["n_init"] * 2, max_iter=CFG["max_iter"],
    tol=CFG["tol"], random_state=RANDOM_STATE,
)
final_gmm.fit(X)

print(f"Converged      : {final_gmm.converged_}")
print(f"Iterations     : {final_gmm.n_iter_}")
print(f"Log-likelihood : {final_gmm.lower_bound_:.6f}")
print(f"BIC            : {final_gmm.bic(X):.2f}")
print(f"AIC            : {final_gmm.aic(X):.2f}")

hard_labels = final_gmm.predict(X)
soft_probs  = final_gmm.predict_proba(X)

df_sessions["gmm_regime"]    = hard_labels
df_sessions["max_posterior"] = soft_probs.max(axis=1)
for k in range(BEST_K):
    df_sessions[f"prob_regime_{k}"] = soft_probs[:, k]

regime_counts = df_sessions["gmm_regime"].value_counts().sort_index()
print("\nRegime distribution:")
for r, c in regime_counts.items():
    print(f"  Regime {r}: {c:>10,}  ({100*c/len(df_sessions):.1f}%)")

joblib.dump(final_gmm,    f"{CFG['output_dir']}/gmm_final_model.pkl")
joblib.dump(fitted_scaler, f"{CFG['output_dir']}/gmm_scaler.pkl")
print("\n[SAVED] gmm_final_model.pkl + gmm_scaler.pkl")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].bar([f"Regime {r}" for r in regime_counts.index], regime_counts.values,
            color=PALETTE[:len(regime_counts)], alpha=0.85, edgecolor="none")
for i, (r, c) in enumerate(regime_counts.items()):
    axes[0].text(i, c + len(df_sessions)*0.005,
                 f"{100*c/len(df_sessions):.1f}%", ha="center", fontsize=9)
axes[0].set_title("Sessions per regime", fontweight="bold")
axes[0].set_ylabel("Count")

axes[1].hist(df_sessions["max_posterior"], bins=60,
             color=PALETTE[2], alpha=0.75, edgecolor="none")
axes[1].axvline(0.8, color="red", ls="--", lw=1.5, label=">0.8 high-confidence")
pct = (df_sessions["max_posterior"] > 0.8).mean() * 100
axes[1].text(0.02, 0.95, f"{pct:.1f}% high-confidence",
             transform=axes[1].transAxes, va="top", fontsize=10)
axes[1].set_xlabel("Max posterior probability")
axes[1].set_title("Assignment confidence", fontweight="bold")
axes[1].legend()
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/05_regime_distribution.png", bbox_inches="tight")
plt.show()
print("[SAVED] 05_regime_distribution.png")

## 8. Segment Profiling & Interpretation

In [ ]:
profile_raw = df_sessions.groupby("gmm_regime")[CFG["features"]].mean()
profile_raw["n_sessions"] = df_sessions.groupby("gmm_regime").size()
profile_raw["pct"]        = profile_raw["n_sessions"] / len(df_sessions) * 100
print("Regime mean profiles (original scale):")
display(profile_raw.round(3))
print("\nComponent weights:")
for k, w in enumerate(final_gmm.weights_):
    print(f"  Regime {k}: {w:.4f}  ({100*w:.1f}%)")
profile_raw.to_csv(f"{CFG['output_dir']}/regime_profiles.csv")
print("[SAVED] regime_profiles.csv")

In [ ]:
def auto_label_regimes(profile, features):
    labels = {}
    ranks  = profile[features].rank(ascending=True, pct=True)
    for k in profile.index:
        r = ranks.loc[k]
        if r.get("session_length_s", 0.5) > 0.7 and r.get("n_events", 0.5) > 0.7:
            label = "Purchase-intent"
        elif r.get("revisit_rate", 0.5) > 0.7 and r.get("session_length_s", 0.5) > 0.4:
            label = "Comparison shopper"
        elif r.get("session_length_s", 0.5) < 0.3 and r.get("n_events", 0.5) < 0.3:
            label = "Cold-start / churn"
        else:
            label = "Casual browser"
        labels[k] = label
    return labels

REGIME_LABELS = auto_label_regimes(profile_raw, CFG["features"])
print("Auto-labels (override after inspection):")
for k, v in REGIME_LABELS.items():
    print(f"  Regime {k} → '{v}'  ({profile_raw.loc[k,'pct']:.1f}%)")

# ── MANUAL OVERRIDE HERE ──────────────────────────────────────────────────────
# REGIME_LABELS = {0: "Casual browser", 1: "Comparison shopper", ...}

df_sessions["regime_label"] = df_sessions["gmm_regime"].map(REGIME_LABELS)
print("\n", df_sessions["regime_label"].value_counts())

In [ ]:
# ── Radar charts ──────────────────────────────────────────────────────────────
p_norm  = profile_raw[CFG["features"]].copy()
p_norm  = (p_norm - p_norm.min()) / (p_norm.max() - p_norm.min() + 1e-9)
n_feat  = len(CFG["features"])
angles  = np.linspace(0, 2*np.pi, n_feat, endpoint=False).tolist() + [0]

fig, axes = plt.subplots(1, BEST_K, figsize=(5*BEST_K, 5),
                          subplot_kw=dict(polar=True))
if BEST_K == 1: axes = [axes]

for i, ax in enumerate(axes):
    vals = p_norm.iloc[i].tolist() + [p_norm.iloc[i].tolist()[0]]
    ax.plot(angles, vals, color=PALETTE[i], lw=2)
    ax.fill(angles, vals, color=PALETTE[i], alpha=0.25)
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([f.replace("_", "\n") for f in CFG["features"]], fontsize=8)
    ax.set_yticklabels([])
    ax.set_title(
        f"Regime {i}: {REGIME_LABELS.get(i,i)}\n({profile_raw.iloc[i]['pct']:.1f}%)",
        fontsize=10, fontweight="bold", pad=15
    )

fig.suptitle("Behavioral profiles (min-max normalised)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/06_regime_radar.png", bbox_inches="tight")
plt.show()
print("[SAVED] 06_regime_radar.png")

In [ ]:
# ── Boxplots by regime ────────────────────────────────────────────────────────
n_cols = 3
n_rows = -(-len(CFG["features"]) // n_cols)
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, n_rows * 4))
axes = axes.flatten()

for i, feat in enumerate(CFG["features"]):
    ax = axes[i]
    bp = ax.boxplot(
        [df_sessions.loc[df_sessions["gmm_regime"] == k, feat].values for k in range(BEST_K)],
        patch_artist=True, showfliers=False,
        medianprops=dict(color="black", lw=2)
    )
    for patch, color in zip(bp["boxes"], PALETTE[:BEST_K]):
        patch.set_facecolor(color); patch.set_alpha(0.7)
    ax.set_title(feat, fontsize=10, fontweight="bold")
    ax.set_xticklabels([f"R{k}" for k in range(BEST_K)])

for j in range(i+1, len(axes)): axes[j].set_visible(False)
fig.suptitle("Feature distributions by GMM regime", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/07_feature_boxplots.png", bbox_inches="tight")
plt.show()
print("[SAVED] 07_feature_boxplots.png")

## 9. PCA + UMAP

In [ ]:
pca    = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca  = pca.fit_transform(X)
plot_n = min(50_000, len(df_sessions))
s_idx  = np.random.choice(len(df_sessions), plot_n, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
ax = axes[0]
for k in range(BEST_K):
    mask = hard_labels[s_idx] == k
    ax.scatter(X_pca[s_idx][mask, 0], X_pca[s_idx][mask, 1],
               s=4, alpha=0.35, color=PALETTE[k], label=f"R{k}: {REGIME_LABELS.get(k,k)}")
    m2d = pca.transform(final_gmm.means_[k:k+1])
    ax.scatter(m2d[0,0], m2d[0,1], s=250, marker="X",
               color=PALETTE[k], edgecolor="black", zorder=10, lw=1.5)
ax.set_title("PCA — hard assignments", fontweight="bold")
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% var)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% var)")
ax.legend(markerscale=3, fontsize=9)

ax = axes[1]
sc = ax.scatter(X_pca[s_idx, 0], X_pca[s_idx, 1],
                c=soft_probs[s_idx].max(axis=1),
                cmap="RdYlGn", s=4, alpha=0.4, vmin=0.3, vmax=1.0)
plt.colorbar(sc, ax=ax, label="Max posterior")
ax.set_title("PCA — confidence", fontweight="bold")
ax.set_xlabel(f"PC1 ({100*pca.explained_variance_ratio_[0]:.1f}% var)")
ax.set_ylabel(f"PC2 ({100*pca.explained_variance_ratio_[1]:.1f}% var)")

plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/08_pca.png", bbox_inches="tight")
plt.show()
print("[SAVED] 08_pca.png")

In [ ]:
if UMAP_AVAILABLE:
    umap_n   = min(50_000, len(df_sessions))
    umap_idx = np.random.choice(len(df_sessions), umap_n, replace=False)
    X_umap   = umap.UMAP(n_components=2, n_neighbors=30, min_dist=0.1,
                          random_state=RANDOM_STATE).fit_transform(X[umap_idx])
    fig, ax = plt.subplots(figsize=(9, 7))
    for k in range(BEST_K):
        mask = hard_labels[umap_idx] == k
        ax.scatter(X_umap[mask, 0], X_umap[mask, 1],
                   s=5, alpha=0.4, color=PALETTE[k], label=f"R{k}: {REGIME_LABELS.get(k,k)}")
    ax.set_title("UMAP — GMM regimes", fontweight="bold")
    ax.legend(markerscale=4)
    plt.tight_layout()
    plt.savefig(f"{CFG['output_dir']}/09_umap.png", bbox_inches="tight")
    plt.show()
    print("[SAVED] 09_umap.png")
else:
    print("[SKIPPED] pip install umap-learn")

## 10. Stability Validation (Bootstrap ARI)

In [ ]:
def bootstrap_stability(X, k, cov_type, n_bootstrap, sample_frac, random_state):
    rng = np.random.default_rng(random_state)
    n   = len(X)
    all_lbl = []
    for _ in tqdm(range(n_bootstrap), desc="Bootstrap"):
        idx = rng.choice(n, int(n * sample_frac), replace=True)
        g   = GaussianMixture(n_components=k, covariance_type=cov_type,
                              n_init=5, max_iter=300,
                              random_state=int(rng.integers(0, 10_000)))
        g.fit(X[idx])
        all_lbl.append(g.predict(X))

    ari = [adjusted_rand_score(all_lbl[i], all_lbl[j])
           for i in range(n_bootstrap) for j in range(i+1, n_bootstrap)]
    return {"mean_ari": float(np.mean(ari)), "std_ari": float(np.std(ari)),
            "min_ari": float(np.min(ari)),   "all_ari": ari}


stability = bootstrap_stability(
    X, BEST_K, CFG["final_cov_type"],
    CFG["bootstrap_n"], CFG["bootstrap_sample_frac"], RANDOM_STATE,
)
print(f"Mean ARI : {stability['mean_ari']:.4f}")
print(f"Std  ARI : {stability['std_ari']:.4f}")
print(f"Min  ARI : {stability['min_ari']:.4f}")
if   stability["mean_ari"] > 0.85: print("✅ HIGH stability")
elif stability["mean_ari"] > 0.60: print("⚠️  MODERATE stability")
else:                               print("❌ LOW stability")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(stability["all_ari"], bins=20, color=PALETTE[3], alpha=0.75, edgecolor="none")
ax.axvline(stability["mean_ari"], color="red", lw=2, ls="--",
           label=f"Mean={stability['mean_ari']:.3f}")
ax.axvline(0.85, color="green", lw=1.5, ls=":", label="Threshold 0.85")
ax.set_xlabel("ARI (pairwise)")
ax.set_title(f"Bootstrap stability — K={BEST_K}", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.savefig(f"{CFG['output_dir']}/10_stability.png", bbox_inches="tight")
plt.show()
print("[SAVED] 10_stability.png")

## 11. Save Outputs

In [ ]:
out_cols = (["user_id", "gmm_regime", "regime_label", "max_posterior"]
            + [f"prob_regime_{k}" for k in range(BEST_K)]
            + CFG["features"])
df_out = df_sessions[[c for c in out_cols if c in df_sessions.columns]]
df_out.to_parquet(f"{CFG['output_dir']}/sessions_with_gmm_labels.parquet", index=False)
df_out.to_csv(f"{CFG['output_dir']}/sessions_with_gmm_labels.csv", index=False)
print(f"[SAVED] sessions_with_gmm_labels  ({len(df_out):,} rows)")

summary = {
    "run_timestamp":       datetime.now().isoformat(),
    "max_total_rows":      CFG["max_total_rows"],
    "n_sessions":          int(len(df_sessions)),
    "n_users":             int(df_sessions["user_id"].nunique()),
    "features_used":       CFG["features"],
    "best_k":              BEST_K,
    "covariance_type":     CFG["final_cov_type"],
    "bic":                 float(final_gmm.bic(X)),
    "aic":                 float(final_gmm.aic(X)),
    "converged":           bool(final_gmm.converged_),
    "stability_mean_ari":  stability["mean_ari"],
    "stability_std_ari":   stability["std_ari"],
    "regime_labels":       {str(k): v for k, v in REGIME_LABELS.items()},
    "regime_sizes":        {str(k): int(v) for k, v in
                            df_sessions["gmm_regime"].value_counts().sort_index().items()},
    "high_confidence_pct": float((df_sessions["max_posterior"] > 0.8).mean()),
}
with open(f"{CFG['output_dir']}/run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)
print("[SAVED] run_summary.json")

print("\n" + "="*60)
for k, v in summary.items():
    if k not in ("features_used",):
        print(f"  {k:35s}: {v}")
print("="*60)